# Шаг 14. Проверка категориальных столбцов

Будет проведена проверка ключевых текстовых полей (manufacturer, era, nationality, decade) на предмет наличия опечаток, разных регистров или синонимов, которые могут повлиять на отображение данных при построении сводных таблиц и дашбордов.

In [18]:
import pandas as pd

# 1. Загружаем очищенный датасет с правильными типами данных
df = pd.read_csv('df_consolidated_clean.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

print("=== 1. Первичный осмотр категориальных столбцов ===")

# Проверяем уникальные значения и частоты для ключевых полей
print("\n--- manufacturer (Топ-10) ---")
print(df['manufacturer'].value_counts().head(10))

print("\n--- era (Все значения) ---")
print(df['era'].value_counts())

print("\n--- nationality (Топ-20) ---")
print(df['nationality'].value_counts().head(20))

print("\n--- decade (Уникальные значения по возрастанию) ---")
print(df['decade'].value_counts().sort_index().head(15))

=== 1. Первичный осмотр категориальных столбцов ===

--- manufacturer (Топ-10) ---
manufacturer
Strelets    561
HaT         455
RedBox      206
Zvezda      191
Mars        188
Caesar      168
Italeri     141
Linear-A    126
Orion       119
Preiser     107
Name: count, dtype: int64

--- era (Все значения) ---
era
Новое время       1072
Новейшее время    1006
Древний мир        639
Средневековье      588
Современность      201
Name: count, dtype: int64

--- nationality (Топ-20) ---
nationality
German         458
Не указано     364
British        348
French         301
Russian        263
USA            262
Italian        257
Spanish         79
Japanese        68
Turkish         62
Austrian        62
Greek           59
Polish          55
Ukrainian       33
Swedish         31
Persian         29
Syrian          25
Egyptian        25
Confederate     22
Iraqi           21
Name: count, dtype: int64

--- decade (Уникальные значения по возрастанию) ---
decade
1950       2
1960      61
1970     12

In [19]:
print("\n=== 2. Стандартизация категориальных значений ===")

# Словарь для очистки национальностей от синонимов и сокращений
# (Мы НЕ трогаем Texian и Confederate, так как для варгеймеров это разные исторические армии)
nat_mapping = {
    'U.S.A.': 'USA',
    'US': 'USA',
    'American': 'USA',
    'UK': 'British',
    'Great Britain': 'British',
    'Holland': 'Dutch',
    'Hollandic': 'Dutch'
}

# Применяем замену
df['nationality'] = df['nationality'].replace(nat_mapping)

# Приводим названия производителей к единому виду (на всякий случай, убираем лишние пробелы)
df['manufacturer'] = df['manufacturer'].str.strip()

# Эры уже стандартизированы на этапе расчёта mid_year, но проверим, нет ли опечаток
# Если есть, можно добавить их в словарь era_mapping

print("Стандартизация завершена.")


=== 2. Стандартизация категориальных значений ===
Стандартизация завершена.


In [20]:
print("\n=== 3. Финальная проверка после стандартизации ===")

# Смотрим, исчезли ли старые варианты
print("Проверка исчезновения 'U.S.A.', 'US', 'American':")
for val in ['U.S.A.', 'US', 'American']:
    count = (df['nationality'] == val).sum()
    print(f"  '{val}': {count} записей")

print("\nОбновленный Топ-15 национальностей:")
print(df['nationality'].value_counts().head(15))

# 4. Сохраняем финальную версию датасета
df.to_csv('df_final.csv', index=False, encoding='utf-8')
print("\n✅ Финальный датасет сохранен как 'df_final.csv'")


=== 3. Финальная проверка после стандартизации ===
Проверка исчезновения 'U.S.A.', 'US', 'American':
  'U.S.A.': 0 записей
  'US': 0 записей
  'American': 0 записей

Обновленный Топ-15 национальностей:
nationality
German        458
Не указано    364
British       348
French        301
Russian       263
USA           262
Italian       257
Spanish        79
Japanese       68
Turkish        62
Austrian       62
Greek          59
Polish         55
Ukrainian      33
Swedish        31
Name: count, dtype: int64

✅ Финальный датасет сохранен как 'df_final.csv'
